In [1]:
# Cell 1
!pip install flask ultralytics opencv-python-headless line-bot-sdk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.8 MB/s eta 0:00:00


In [8]:
# Cell 2:下載模型、初始化
import os
import urllib.request

MODEL_URL = "https://github.com/jonwang5150/drowning_detection/raw/refs/heads/main/best.pt"
# MODEL_URL = "https://github.com/jonwang5150/drowning_detection/raw/refs/heads/main/best_new.pt"

MODEL_PATH = "/content/best.pt"
# MODEL_PATH = "/content/best_new.pt"


if not os.path.exists(MODEL_PATH):
    print("正在從 GitHub 下載模型...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"模型已下載至: {MODEL_PATH}")
else:
    print("模型已存在，略過下載")

FIXED_TRACK_BUFFER = 150
TRACKER_YAML = "/content/custom_bytetrack.yaml"
with open(TRACKER_YAML, "w") as f:
    f.write(f"""tracker_type: bytetrack
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.25
track_buffer: {FIXED_TRACK_BUFFER}
match_thresh: 0.8
fuse_score: True
""")
print(f"Track Buffer 已固定設定為: {FIXED_TRACK_BUFFER}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"運算裝置: {device}")

from ultralytics import YOLO
model = YOLO(MODEL_PATH)
model.to(device)
print("模型類別:", model.names)


模型已存在，略過下載
Track Buffer 已固定設定為: 150
運算裝置: cuda
模型類別: {0: 'Drowning', 1: 'Person out of water', 2: 'Swimming'}


In [9]:
# Cell 3
import cv2
import time
import threading
import csv
import io
import os
import numpy as np
from datetime import datetime
from collections import defaultdict, deque

from flask import Flask, Response, request, jsonify, render_template_string, send_file

from linebot import LineBotApi
from linebot.models import TextSendMessage

TARGET_CLASSES = ["Drowning", "Person out of water", "Swimming"]
DISPLAY_MAX_WIDTH = 720
UPLOAD_PATH = "/content/uploaded_video.mp4"
LOG_PATH = "/content/detection_log.csv"

app = Flask(__name__)

_black_frame = np.zeros((360, 640, 3), dtype=np.uint8)
cv2.putText(_black_frame, "No video loaded", (170, 180),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (150, 150, 150), 2)
_, _black_buffer = cv2.imencode('.jpg', _black_frame)
BLACK_FRAME_JPEG = _black_buffer.tobytes()


# ========================================================
# 全域狀態管理
# ========================================================
class DetectionState:
    def __init__(self):
        self.lock = threading.Lock()
        self.latest_frame_jpeg = None
        self.stats_text = "尚未開始處理"
        self.warning_active = False
        self.running = False
        self.paused = False
        self.thread = None
        self.file_uploaded = False
        self.log_ready = False

        self.line_enabled = False
        self.line_token = ""

    def update_frame(self, jpeg_bytes):
        with self.lock:
            self.latest_frame_jpeg = jpeg_bytes

    def clear_frame(self):
        with self.lock:
            self.latest_frame_jpeg = None

    def update_stats(self, text, warning):
        with self.lock:
            self.stats_text = text
            self.warning_active = warning

    def get_frame(self):
        with self.lock:
            return self.latest_frame_jpeg

    def get_stats(self):
        with self.lock:
            return self.stats_text, self.warning_active, self.paused, self.running, self.log_ready


state = DetectionState()


def fmt_time(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m:02d}:{s:02d}"


def extract_first_frame_jpeg(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None

    h, w = frame.shape[:2]
    if w > DISPLAY_MAX_WIDTH:
        scale = DISPLAY_MAX_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    ok, buffer = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return buffer.tobytes() if ok else None


def draw_dashboard(frame, fps, counts, active_warning_count, warning_trigger_total):
    panel_w, panel_h = 360, 250
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    x, y = 15, 35
    line_h = 36

    cv2.putText(frame, f"FPS: {fps:.1f}", (x, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 255, 255), 2)
    y += line_h + 5
    cv2.line(frame, (x, y - 18), (panel_w - 15, y - 18), (100, 100, 100), 1)

    color_map = {
        "Drowning": (0, 0, 255),
        "Person out of water": (0, 165, 255),
        "Swimming": (0, 255, 0),
    }
    for cls_name in TARGET_CLASSES:
        count = counts.get(cls_name, 0)
        color = color_map.get(cls_name, (255, 255, 255))
        cv2.putText(frame, f"{cls_name}: {count}", (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2)
        y += line_h

    alert_color = (0, 0, 255) if active_warning_count > 0 else (200, 200, 200)
    cv2.putText(frame, f"Active Alerts: {active_warning_count}", (x, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, alert_color, 2)
    y += line_h

    cv2.putText(frame, f"Possible Drowning: {warning_trigger_total}", (x, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 0, 255), 2)

    return frame


def draw_warning_banner(frame, w):
    text = "!! POSSIBLE DROWNING !!"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.3
    thickness = 3
    (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)
    banner_h = text_h + 40
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, banner_h), (0, 0, 200), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
    text_x = (w - text_w) // 2
    text_y = banner_h - 20
    cv2.putText(frame, text, (text_x, text_y), font, font_scale, (0, 0, 0), thickness + 3)
    cv2.putText(frame, text, (text_x, text_y), font, font_scale, (255, 255, 255), thickness)
    return frame


def resolve_source(raw_source, is_stream):
    if not is_stream:
        return raw_source
    raw_source = raw_source.strip()
    if raw_source.isdigit():
        return int(raw_source)
    return raw_source


# ========================================================
# LINE 通知：純文字，背景執行緒送出避免阻塞主處理迴圈
# ========================================================
def send_line_text_async(token, message_text):
    def _send():
        try:
            line_bot_api = LineBotApi(token)
            line_bot_api.broadcast(TextSendMessage(text=message_text))
            print(f"[LINE] 通知已送出: {message_text}")
        except Exception as e:
            print(f"[LINE] 發送失敗: {e}")

    threading.Thread(target=_send, daemon=True).start()


# ========================================================
# 背景執行緒：處理影片/串流
# ========================================================
def processing_worker(source, is_stream, conf_threshold, warning_ratio,
                       warning_window_sec, min_ratio_of_window):
    cap = None
    csv_file = None
    csv_writer = None

    line_enabled = state.line_enabled
    line_token = state.line_token

    try:
        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            state.update_stats(f"無法開啟來源：{source}", False)
            return

        fps_video = cap.get(cv2.CAP_PROP_FPS) or 30
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total_sec = total_frames / fps_video if fps_video and total_frames > 0 else 0

        warning_window_frames = max(1, int(round(warning_window_sec * fps_video)))
        min_samples_required = max(1, int(round(warning_window_frames * min_ratio_of_window)))

        track_history = defaultdict(lambda: deque(maxlen=warning_window_frames))
        active_warning_ids = set()
        warned_ids_ever = set()
        warning_trigger_total = 0
        target_counts = {c: 0 for c in TARGET_CLASSES}

        csv_file = open(LOG_PATH, "w", newline="", encoding="utf-8-sig")
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow([
            "系統時間", "影片時間", "幀編號", "Tracking_ID", "檢測類別",
            "信心度(%)", "邊界框_X1", "邊界框_Y1", "邊界框_X2", "邊界框_Y2"
        ])
        state.log_ready = False

        frame_idx = 0
        prev_time = time.time()
        smoothed_fps = 0.0

        while state.running:
            if state.paused:
                time.sleep(0.1)
                continue

            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            now = time.time()
            instant_fps = 1.0 / max(now - prev_time, 1e-6)
            smoothed_fps = instant_fps if smoothed_fps == 0 else (0.9 * smoothed_fps + 0.1 * instant_fps)
            prev_time = now

            results = model.track(
                frame, conf=conf_threshold, device=device,
                persist=True, tracker=TRACKER_YAML, verbose=False
            )
            result = results[0]

            active_warning_ids.clear()
            newly_triggered_ids = []

            wall_clock_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            video_time_str = fmt_time(frame_idx / fps_video) if (not is_stream and fps_video) else ""

            if result.boxes is not None:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = model.names[cls_id]
                    conf_pct = float(box.conf[0]) * 100
                    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
                    tid = int(box.id[0]) if box.id is not None else -1

                    if cls_name in target_counts:
                        target_counts[cls_name] += 1

                    csv_writer.writerow([
                        wall_clock_now, video_time_str, frame_idx,
                        tid, cls_name, f"{conf_pct:.1f}",
                        f"{x1:.1f}", f"{y1:.1f}", f"{x2:.1f}", f"{y2:.1f}"
                    ])

                    if box.id is not None:
                        track_history[tid].append(cls_name)
                        dq = track_history[tid]
                        total = len(dq)
                        if total >= min_samples_required:
                            drowning_count = sum(1 for c in dq if c == "Drowning")
                            ratio = drowning_count / total
                            if ratio > warning_ratio:
                                active_warning_ids.add(tid)
                                if tid not in warned_ids_ever:
                                    warned_ids_ever.add(tid)
                                    warning_trigger_total += 1
                                    newly_triggered_ids.append(tid)

            if frame_idx % 30 == 0:
                csv_file.flush()

            annotated = result.plot()

            if result.boxes is not None:
                for box in result.boxes:
                    if box.id is not None and int(box.id[0]) in active_warning_ids:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])
                        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 4)
                        cv2.putText(annotated, "WARNING", (x1, max(0, y1 - 10)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            annotated = draw_dashboard(annotated, smoothed_fps, target_counts,
                                        len(active_warning_ids), warning_trigger_total)
            if len(active_warning_ids) > 0:
                annotated = draw_warning_banner(annotated, width)

            # ★ 純文字通知：只在首次觸發時發送
            if newly_triggered_ids and line_enabled and line_token:
                if not is_stream:
                    time_desc = f"影片時間 {fmt_time(frame_idx / fps_video)}"
                else:
                    time_desc = f"系統時間 {datetime.now().strftime('%H:%M:%S')}"

                msg = (
                    f"⚠️ 發現疑似溺水者\n"
                    f"{time_desc}\n"
                    f"幀編號: {frame_idx}\n"
                    f"Tracking ID: {', '.join(map(str, newly_triggered_ids))}"
                )
                send_line_text_async(line_token, msg)

            disp_frame = annotated
            if width > DISPLAY_MAX_WIDTH:
                scale = DISPLAY_MAX_WIDTH / width
                disp_frame = cv2.resize(annotated, (int(width * scale), int(height * scale)))

            ok, buffer = cv2.imencode('.jpg', disp_frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
            if ok:
                state.update_frame(buffer.tobytes())

            cur_sec = frame_idx / fps_video if fps_video else 0
            display_clock = datetime.now().strftime("%H:%M:%S")
            if not is_stream:
                time_line = f"影片時間: {fmt_time(cur_sec)} / {fmt_time(total_sec)} | 系統時間: {display_clock}"
            else:
                time_line = f"串流已運行: {fmt_time(cur_sec)} | 系統時間: {display_clock}"

            progress = f"{frame_idx}/{total_frames}" if not is_stream else f"{frame_idx} (即時串流)"
            stats = (
                f"{time_line}\n"
                f"處理進度: {progress} | FPS: {smoothed_fps:.1f}\n"
                f"{'-'*35}\n"
            )
            for name in TARGET_CLASSES:
                stats += f"{name}: {target_counts.get(name, 0)}\n"
            stats += f"{'-'*35}\n累計警示物件數: {warning_trigger_total}"

            state.update_stats(stats, len(active_warning_ids) > 0)

        final_text, _, _, _, _ = state.get_stats()
        if "===== 處理完成 =====" not in final_text and "無法開啟" not in final_text:
            state.update_stats(final_text + "\n\n===== 處理已結束 =====", False)

    finally:
        if cap is not None:
            cap.release()
        if csv_file is not None:
            csv_file.flush()
            csv_file.close()
            state.log_ready = True
        state.running = False
        state.paused = False


# ========================================================
# Flask 路由
# ========================================================
@app.route('/')
def index():
    return render_template_string(HTML_PAGE)


@app.route('/video_feed')
def video_feed():
    def generate():
        while True:
            frame = state.get_frame()
            if frame is None:
                frame = BLACK_FRAME_JPEG
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame + b'\r\n')
            time.sleep(0.03)
    return Response(generate(), mimetype='multipart/x-mixed-replace; boundary=frame')


@app.route('/stats')
def stats():
    text, warning, paused, running, log_ready = state.get_stats()
    return jsonify({"stats": text, "warning": warning, "running": running,
                     "paused": paused, "log_ready": log_ready})


@app.route('/upload', methods=['POST'])
def upload():
    if state.running:
        return jsonify({"ok": False, "msg": "處理中無法上傳新檔案，請先按停止"})

    file = request.files.get('video_file')
    if not file or file.filename == '':
        return jsonify({"ok": False, "msg": "請選擇影片檔案"})

    file.save(UPLOAD_PATH)
    state.file_uploaded = True

    first_frame_jpeg = extract_first_frame_jpeg(UPLOAD_PATH)
    if first_frame_jpeg is not None:
        state.update_frame(first_frame_jpeg)
    else:
        state.clear_frame()

    state.update_stats("新影片已上傳，尚未開始處理", False)
    state.log_ready = False

    return jsonify({"ok": True, "msg": "上傳完成"})


@app.route('/start', methods=['POST'])
def start():
    if state.thread is not None and state.thread.is_alive():
        state.running = False
        state.thread.join(timeout=10)

    source_type = request.form.get('source_type')
    conf_threshold = float(request.form.get('conf_threshold', 0.25))
    warning_ratio = float(request.form.get('warning_ratio', 0.6))
    warning_window_sec = float(request.form.get('warning_window_sec', 5.0))
    min_ratio_of_window = float(request.form.get('min_ratio_of_window', 0.3))

    state.line_enabled = request.form.get('line_enabled', 'false') == 'true'
    state.line_token = request.form.get('line_token', '').strip()

    if state.line_enabled and not state.line_token:
        return jsonify({"ok": False, "msg": "已勾選LINE通知，但尚未輸入CHANNEL_ACCESS_TOKEN"})

    if source_type == 'upload':
        if not state.file_uploaded or not os.path.exists(UPLOAD_PATH):
            return jsonify({"ok": False, "msg": "請先上傳影片檔案"})
        source = UPLOAD_PATH
        is_stream = False
    else:
        raw_stream = request.form.get('stream_url', '').strip()
        if not raw_stream:
            return jsonify({"ok": False, "msg": "請輸入串流網址或攝影機編號"})
        source = resolve_source(raw_stream, True)
        is_stream = True

    model.predictor = None

    state.running = True
    state.paused = False
    state.log_ready = False
    state.update_stats("啟動中...", False)

    state.thread = threading.Thread(
        target=processing_worker,
        args=(source, is_stream, conf_threshold, warning_ratio,
              warning_window_sec, min_ratio_of_window),
        daemon=True
    )
    state.thread.start()
    return jsonify({"ok": True, "msg": "已開始處理"})


@app.route('/stop', methods=['POST'])
def stop():
    state.running = False
    state.paused = False
    if state.thread is not None and state.thread.is_alive():
        state.thread.join(timeout=10)
    return jsonify({"ok": True, "msg": "已停止"})


@app.route('/pause', methods=['POST'])
def pause():
    if not state.running:
        return jsonify({"ok": False, "msg": "目前沒有正在執行的任務"})
    state.paused = True
    return jsonify({"ok": True, "msg": "已暫停"})


@app.route('/resume', methods=['POST'])
def resume():
    if not state.running:
        return jsonify({"ok": False, "msg": "目前沒有正在執行的任務"})
    state.paused = False
    return jsonify({"ok": True, "msg": "已繼續"})


@app.route('/download_log')
def download_log():
    if not os.path.exists(LOG_PATH):
        return jsonify({"ok": False, "msg": "尚無可下載的log"}), 404
    return send_file(LOG_PATH, as_attachment=True,
                      download_name=f"detection_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")


@app.route('/test_line', methods=['POST'])
def test_line():
    token = request.form.get('line_token', '').strip()
    if not token:
        return jsonify({"ok": False, "msg": "請先輸入CHANNEL_ACCESS_TOKEN"})
    try:
        line_bot_api = LineBotApi(token)
        line_bot_api.broadcast(TextSendMessage(text="✅ 溺水偵測系統：測試訊息，LINE通知功能已成功連接。"))
        return jsonify({"ok": True, "msg": "測試訊息已送出，請確認LINE官方帳號是否收到"})
    except Exception as e:
        return jsonify({"ok": False, "msg": f"發送失敗: {e}"})


# ========================================================
# 前端頁面
# ========================================================
HTML_PAGE = """
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<title>溺水偵測系統</title>
<style>
    body { font-family: -apple-system, "Segoe UI", Arial, sans-serif; background:#0f1116; color:#e6e6e6; margin:0; padding:20px; }
    h1 { font-size: 22px; margin-bottom: 4px; }
    .subtitle { color:#9aa0a6; margin-bottom: 20px; font-size: 13px; }
    .container { display:flex; gap: 20px; }
    .left { flex: 2; }
    .right { flex: 1; background:#1a1d24; border-radius:10px; padding:16px; }
    #videoImg { width:100%; border-radius:10px; background:#000; display:block; }
    .warning-banner { display:none; background:#d32f2f; color:white; text-align:center;
                       padding:10px; font-weight:bold; font-size:18px; border-radius:6px; margin-top:8px; }
    .warning-banner.show { display:block; animation: pulse 1s infinite; }
    @keyframes pulse { 0%{opacity:1;} 50%{opacity:0.6;} 100%{opacity:1;} }
    #statsBox { background:#1a1d24; border-radius:10px; padding:14px; margin-top:10px;
                white-space:pre-wrap; font-family: Consolas, monospace; font-size:13px; min-height:140px; }
    label { display:block; margin-top:14px; margin-bottom:4px; font-size:13px; color:#c9ccd1; }
    input[type=range] { width:100%; }
    input[type=text], input[type=password] { width:100%; padding:6px; border-radius:6px; border:1px solid #333;
                        background:#0f1116; color:#e6e6e6; box-sizing:border-box; }
    .radio-group label { display:inline-block; margin-right:16px; font-size:14px; }
    button { padding:10px; margin-top:10px; border:none; border-radius:8px;
             font-size:14px; cursor:pointer; font-weight:600; }
    button:disabled { opacity:0.4; cursor:not-allowed; }
    input:disabled { opacity:0.4; cursor:not-allowed; }
    .btn-row { display:flex; gap:8px; }
    .btn-row button { flex:1; }
    .btn-start { background:#4CAF50; color:white; width:100%; }
    .btn-pause { background:#FF9800; color:white; }
    .btn-stop { background:#555; color:white; }
    .btn-upload { background:#2196F3; color:white; white-space:nowrap; }
    .btn-cancel { background:#f44336; color:white; white-space:nowrap; }
    .btn-reset { background:#607D8B; color:white; width:100%; }
    .btn-download { background:#009688; color:white; width:100%; text-decoration:none;
                     text-align:center; display:block; box-sizing:border-box; }
    .btn-test-line { background:#06C755; color:white; width:100%; }
    .slider-val { float:right; color:#8ab4f8; }
    .section-title { font-weight:700; margin-top:4px; }
    .hint { color:#9aa0a6; font-size:11px; margin-top:6px; }

    .file-row { display:flex; gap:8px; align-items:center; }
    .file-row input[type=file] { flex:1; padding:6px; border-radius:6px; border:1px solid #333;
                                  background:#0f1116; color:#e6e6e6; box-sizing:border-box; }

    .upload-status { margin-top:8px; font-size:12px; color:#8ab4f8; min-height:16px; }
    .upload-status.success { color:#4CAF50; }
    .upload-status.error { color:#f44336; }
    .progress-bar-bg { width:100%; background:#333; border-radius:6px; height:8px; margin-top:4px; overflow:hidden; display:none; }
    .progress-bar-fill { height:100%; background:#4CAF50; width:0%; transition: width 0.15s; }

    .locked-hint { display:none; color:#FF9800; font-size:12px; margin-top:8px; padding:8px;
                   background:rgba(255,152,0,0.1); border-radius:6px; }
    .locked-hint.show { display:block; }

    .line-section { border-top:1px solid #333; margin-top:16px; padding-top:12px; }
    .line-status { font-size:12px; margin-top:6px; min-height:16px; }
    .checkbox-row { display:flex; align-items:center; gap:8px; margin-top:10px; }
    .checkbox-row label { margin:0; }
    .checkbox-row input[type=checkbox] { width:16px; height:16px; }
</style>
</head>
<body>

<h1>🏊 溺水偵測系統</h1>
<div class="subtitle">Flask + MJPEG 串流版</div>

<div class="container">
    <div class="left">
        <img id="videoImg" src="/video_feed">
        <div id="warningBanner" class="warning-banner">⚠️ POSSIBLE DROWNING ⚠️</div>
        <div id="statsBox">尚未開始處理</div>
        <a class="btn-download" id="downloadBtn" href="/download_log" style="display:none;">⬇ 下載偵測紀錄 (CSV)</a>
    </div>

    <div class="right">
        <div class="section-title">影片來源</div>
        <div class="radio-group">
            <label><input type="radio" name="source_type" value="upload" checked onchange="toggleSource()" class="ctrl-input"> 上傳影片</label>
            <label><input type="radio" name="source_type" value="stream" onchange="toggleSource()" class="ctrl-input"> 視訊串流</label>
        </div>

        <div id="uploadBlock">
            <label>選擇影片檔案</label>
            <div class="file-row">
                <input type="file" id="videoFile" accept="video/*" class="ctrl-input" onchange="onFileSelected()">
                <button class="btn-upload ctrl-input" id="uploadBtn" onclick="uploadFile()" disabled>⬆ 上傳</button>
                <button class="btn-cancel" id="cancelBtn" onclick="cancelUpload()" style="display:none;">✕ 取消</button>
            </div>
            <div class="progress-bar-bg" id="progressBarBg">
                <div class="progress-bar-fill" id="progressBarFill"></div>
            </div>
            <div class="upload-status" id="uploadStatus"></div>
        </div>

        <div id="streamBlock" style="display:none;">
            <label>攝影機編號 或 串流網址</label>
            <input type="text" id="streamUrl" class="ctrl-input"
                   placeholder="例如 0（本機攝影機）或 rtsp://192.168.1.100:554/stream">
            <div class="hint">
                純數字（如 0、1）＝讀取本機攝影機裝置編號（僅限本機執行有效，Colab雲端環境沒有實體攝影機）<br>
                RTSP/HTTP網址＝連接遠端串流來源（目前尚未實測）
            </div>
        </div>

        <label>信心門檻 (conf_threshold) <span class="slider-val" id="confVal">0.25</span></label>
        <input type="range" id="confSlider" class="ctrl-input" min="0.05" max="0.95" step="0.01" value="0.25"
               oninput="document.getElementById('confVal').innerText=this.value">

        <label>Drowning 佔比門檻 <span class="slider-val" id="ratioVal">0.60</span></label>
        <input type="range" id="ratioSlider" class="ctrl-input" min="0.1" max="1.0" step="0.05" value="0.6"
               oninput="document.getElementById('ratioVal').innerText=this.value">

        <label>觀察視窗秒數 <span class="slider-val" id="windowVal">5.0</span></label>
        <input type="range" id="windowSlider" class="ctrl-input" min="1" max="10" step="0.5" value="5"
               oninput="document.getElementById('windowVal').innerText=this.value">

        <label>最低樣本比例 <span class="slider-val" id="minRatioVal">0.30</span></label>
        <input type="range" id="minRatioSlider" class="ctrl-input" min="0.05" max="1.0" step="0.05" value="0.3"
               oninput="document.getElementById('minRatioVal').innerText=this.value">

        <button class="btn-reset ctrl-input" id="resetBtn" onclick="resetDefaults()">↺ 回復預設值</button>

        <div class="line-section">
            <div class="section-title">LINE 通知（純文字）</div>
            <div class="checkbox-row">
                <input type="checkbox" id="lineEnabled" class="ctrl-input" onchange="toggleLineFields()">
                <label style="margin:0;">啟用溺水通知（透過LINE官方帳號廣播文字訊息）</label>
            </div>

            <div id="lineFields" style="display:none;">
                <label>CHANNEL_ACCESS_TOKEN</label>
                <input type="password" id="lineToken" class="ctrl-input" placeholder="貼上你的LINE Channel Access Token">
                <div class="hint">
                    通知內容包含：時間（影片時間或系統時間）、幀編號、Tracking ID，僅發送文字，不含截圖。
                </div>

                <button class="btn-test-line ctrl-input" onclick="testLine()">🔔 測試發送</button>
                <div class="line-status" id="lineTestStatus"></div>
            </div>
        </div>

        <div class="locked-hint" id="lockedHint">🔒 處理中，參數已鎖定。按「停止」後才能調整並重新開始。</div>

        <div class="hint">※ 參數需在按下「開始處理」前設定好</div>

        <button class="btn-start" id="startBtn" onclick="startProcessing()">▶ 開始處理</button>
        <div class="btn-row">
            <button class="btn-pause" id="pauseBtn" onclick="togglePause()" disabled>⏸ 暫停</button>
            <button class="btn-stop" id="stopBtn" onclick="stopProcessing()" disabled>⏹ 停止</button>
        </div>
    </div>
</div>

<script>
let isPaused = false;
let isProcessing = false;
let currentXhr = null;

const DEFAULTS = { conf: 0.25, ratio: 0.6, window: 5.0, minRatio: 0.3 };

function resetDefaults() {
    document.getElementById('confSlider').value = DEFAULTS.conf;
    document.getElementById('confVal').innerText = DEFAULTS.conf.toFixed(2);
    document.getElementById('ratioSlider').value = DEFAULTS.ratio;
    document.getElementById('ratioVal').innerText = DEFAULTS.ratio.toFixed(2);
    document.getElementById('windowSlider').value = DEFAULTS.window;
    document.getElementById('windowVal').innerText = DEFAULTS.window.toFixed(1);
    document.getElementById('minRatioSlider').value = DEFAULTS.minRatio;
    document.getElementById('minRatioVal').innerText = DEFAULTS.minRatio.toFixed(2);
}

function toggleSource() {
    const type = document.querySelector('input[name=source_type]:checked').value;
    document.getElementById('uploadBlock').style.display = (type === 'upload') ? 'block' : 'none';
    document.getElementById('streamBlock').style.display = (type === 'stream') ? 'block' : 'none';
}

function toggleLineFields() {
    const enabled = document.getElementById('lineEnabled').checked;
    document.getElementById('lineFields').style.display = enabled ? 'block' : 'none';
}

function testLine() {
    const token = document.getElementById('lineToken').value.trim();
    if (!token) { alert('請先輸入CHANNEL_ACCESS_TOKEN'); return; }

    const statusEl = document.getElementById('lineTestStatus');
    statusEl.style.color = '#8ab4f8';
    statusEl.innerText = '發送中...';

    const formData = new FormData();
    formData.append('line_token', token);

    fetch('/test_line', { method: 'POST', body: formData })
        .then(r => r.json())
        .then(data => {
            statusEl.style.color = data.ok ? '#4CAF50' : '#f44336';
            statusEl.innerText = (data.ok ? '✅ ' : '❌ ') + data.msg;
        });
}

function onFileSelected() {
    const fileInput = document.getElementById('videoFile');
    const uploadBtn = document.getElementById('uploadBtn');
    uploadBtn.disabled = (fileInput.files.length === 0);
    document.getElementById('uploadStatus').innerText = '';
    document.getElementById('uploadStatus').className = 'upload-status';
    document.getElementById('progressBarBg').style.display = 'none';
    document.getElementById('progressBarFill').style.width = '0%';
}

function uploadFile() {
    const fileInput = document.getElementById('videoFile');
    if (fileInput.files.length === 0) { alert('請先選擇影片檔案'); return; }

    const formData = new FormData();
    formData.append('video_file', fileInput.files[0]);

    const xhr = new XMLHttpRequest();
    currentXhr = xhr;

    const uploadBtn = document.getElementById('uploadBtn');
    const cancelBtn = document.getElementById('cancelBtn');
    const progressBarBg = document.getElementById('progressBarBg');
    const progressBarFill = document.getElementById('progressBarFill');
    const uploadStatus = document.getElementById('uploadStatus');

    uploadBtn.disabled = true;
    cancelBtn.style.display = 'inline-block';
    progressBarBg.style.display = 'block';
    progressBarFill.style.width = '0%';
    uploadStatus.className = 'upload-status';
    uploadStatus.innerText = '準備上傳...';

    xhr.upload.onprogress = function(e) {
        if (e.lengthComputable) {
            const pct = Math.round((e.loaded / e.total) * 100);
            progressBarFill.style.width = pct + '%';
            uploadStatus.innerText = `上傳中... ${pct}%`;
        }
    };

    xhr.onload = function() {
        cancelBtn.style.display = 'none';
        currentXhr = null;
        if (xhr.status === 200) {
            const data = JSON.parse(xhr.responseText);
            if (data.ok) {
                uploadStatus.className = 'upload-status success';
                uploadStatus.innerText = '✅ 上傳完成，已顯示首幀，可以按「開始處理」';
                progressBarFill.style.width = '100%';
                document.getElementById('videoImg').src = '/video_feed?t=' + Date.now();
                document.getElementById('downloadBtn').style.display = 'none';
                uploadBtn.disabled = true;
            } else {
                uploadStatus.className = 'upload-status error';
                uploadStatus.innerText = '❌ ' + data.msg;
                alert(data.msg);
                uploadBtn.disabled = false;
            }
        } else {
            uploadStatus.className = 'upload-status error';
            uploadStatus.innerText = '❌ 上傳失敗';
            uploadBtn.disabled = false;
        }
    };

    xhr.onerror = function() {
        cancelBtn.style.display = 'none';
        currentXhr = null;
        uploadStatus.className = 'upload-status error';
        uploadStatus.innerText = '❌ 網路錯誤，上傳失敗';
        uploadBtn.disabled = false;
    };

    xhr.onabort = function() {
        cancelBtn.style.display = 'none';
        currentXhr = null;
        uploadStatus.className = 'upload-status';
        uploadStatus.innerText = '已取消上傳';
        progressBarFill.style.width = '0%';
        progressBarBg.style.display = 'none';
        uploadBtn.disabled = false;
    };

    xhr.open('POST', '/upload', true);
    xhr.send(formData);
}

function cancelUpload() {
    if (currentXhr) {
        currentXhr.abort();
    }
}

function startProcessing() {
    const sourceType = document.querySelector('input[name=source_type]:checked').value;
    const formData = new FormData();
    formData.append('source_type', sourceType);
    formData.append('conf_threshold', document.getElementById('confSlider').value);
    formData.append('warning_ratio', document.getElementById('ratioSlider').value);
    formData.append('warning_window_sec', document.getElementById('windowSlider').value);
    formData.append('min_ratio_of_window', document.getElementById('minRatioSlider').value);

    const lineEnabled = document.getElementById('lineEnabled').checked;
    formData.append('line_enabled', lineEnabled ? 'true' : 'false');
    formData.append('line_token', document.getElementById('lineToken').value.trim());

    if (sourceType === 'stream') {
        formData.append('stream_url', document.getElementById('streamUrl').value);
    }

    document.getElementById('downloadBtn').style.display = 'none';

    fetch('/start', { method: 'POST', body: formData })
        .then(r => r.json())
        .then(data => {
            if (data.ok) {
                setLocked(true);
            } else {
                alert(data.msg);
            }
        });
}

function stopProcessing() {
    fetch('/stop', { method: 'POST' })
        .then(r => r.json())
        .then(data => {
            console.log(data.msg);
            isPaused = false;
            document.getElementById('pauseBtn').innerText = '⏸ 暫停';
            setLocked(false);
        });
}

function togglePause() {
    const endpoint = isPaused ? '/resume' : '/pause';
    fetch(endpoint, { method: 'POST' })
        .then(r => r.json())
        .then(data => {
            if (data.ok) {
                isPaused = !isPaused;
                document.getElementById('pauseBtn').innerText = isPaused ? '▶ 繼續' : '⏸ 暫停';
            } else {
                alert(data.msg);
            }
        });
}

function setLocked(locked) {
    isProcessing = locked;
    document.querySelectorAll('.ctrl-input').forEach(el => { el.disabled = locked; });
    document.getElementById('startBtn').disabled = locked;
    document.getElementById('lockedHint').classList.toggle('show', locked);
    document.getElementById('pauseBtn').disabled = !locked;
    document.getElementById('stopBtn').disabled = !locked;

    if (locked) {
        document.getElementById('uploadBtn').disabled = true;
    }
}

setInterval(() => {
    fetch('/stats')
        .then(r => r.json())
        .then(data => {
            document.getElementById('statsBox').innerText = data.stats;
            const banner = document.getElementById('warningBanner');
            if (data.warning) { banner.classList.add('show'); }
            else { banner.classList.remove('show'); }

            isPaused = data.paused;
            document.getElementById('pauseBtn').innerText = isPaused ? '▶ 繼續' : '⏸ 暫停';

            if (data.running !== isProcessing) {
                setLocked(data.running);
            }

            document.getElementById('downloadBtn').style.display = data.log_ready ? 'block' : 'none';
        });
}, 1000);

setLocked(false);
</script>

</body>
</html>
"""

print("Flask app 已定義完成")

Flask app 已定義完成


In [10]:
# Cell 4:啟動伺服器(用Colab內建通道,不需要ngrok帳號)
import threading
from google.colab import output

def run_flask():
    app.run(host='0.0.0.0', port=5000, threaded=True)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

import time
time.sleep(2)  # 等伺服器啟動

output.serve_kernel_port_as_window(5000)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>